# Support Vector Machines — from scratch

An SVM finds the hyperplane $\mathbf{w}^\top\mathbf{x} + b = 0$ that separates two classes ($y\in\{-1,+1\}$)
with the **largest margin** $\frac{2}{\lVert\mathbf{w}\rVert}$.

### Soft-margin primal
$$\min_{\mathbf{w},b}\; \frac{1}{2}\lVert\mathbf{w}\rVert^2 + C\sum_i \max\big(0,\; 1 - y_i(\mathbf{w}^\top\mathbf{x}_i + b)\big)$$
The second term is the **hinge loss**. $C$ trades margin width against training errors.
We solve this with (sub)gradient descent — **Part 1**.

### Dual & kernels
$$\max_{\alpha}\; \sum_i\alpha_i - \frac{1}{2}\sum_{i,j}\alpha_i\alpha_j y_i y_j K(\mathbf{x}_i,\mathbf{x}_j) \quad\text{s.t. } 0\le\alpha_i\le C,\; \sum_i\alpha_i y_i = 0$$
Prediction: $f(\mathbf{x}) = \sum_i \alpha_i y_i K(\mathbf{x}_i, \mathbf{x}) + b$. Only points with $\alpha_i > 0$
(**support vectors**) matter. Replacing the dot product with a **kernel** gives non-linear boundaries:

* Linear: $K(\mathbf{a},\mathbf{b}) = \mathbf{a}^\top\mathbf{b}$
* Polynomial: $K = (\mathbf{a}^\top\mathbf{b} + c)^d$
* RBF: $K = \exp(-\gamma\lVert\mathbf{a}-\mathbf{b}\rVert^2)$

We solve the dual with **SMO** (Sequential Minimal Optimisation) — **Part 2**.

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

# NumPy 2.0 with Apple's Accelerate BLAS raises spurious "encountered in matmul" warnings on macOS
warnings.filterwarnings("ignore", message=".*encountered in matmul")

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 5)

In [ ]:
def make_blobs(n_samples=300, centers=3, cluster_std=1.0, n_features=2,
               center_box=(-10, 10), random_state=None):
    """Isotropic Gaussian blobs (re-implementation of sklearn's make_blobs)."""
    rng = np.random.RandomState(random_state)
    if isinstance(centers, int):
        centers = rng.uniform(*center_box, size=(centers, n_features))
    centers = np.asarray(centers, dtype=float)
    k = len(centers)
    counts = [n_samples // k + (1 if i < n_samples % k else 0) for i in range(k)]
    stds = np.broadcast_to(np.asarray(cluster_std, dtype=float), (k,))
    X = np.vstack([rng.normal(c, s, size=(m, centers.shape[1]))
                   for c, s, m in zip(centers, stds, counts)])
    y = np.concatenate([np.full(m, i) for i, m in enumerate(counts)])
    idx = rng.permutation(n_samples)
    return X[idx], y[idx]

def make_moons(n_samples=200, noise=0.1, random_state=None):
    """Two interleaving half circles."""
    rng = np.random.RandomState(random_state)
    n_out = n_samples // 2
    n_in = n_samples - n_out
    t_out = np.linspace(0, np.pi, n_out)
    t_in = np.linspace(0, np.pi, n_in)
    outer = np.c_[np.cos(t_out), np.sin(t_out)]
    inner = np.c_[1 - np.cos(t_in), 0.5 - np.sin(t_in)]
    X = np.vstack([outer, inner]) + rng.normal(0, noise, size=(n_samples, 2))
    y = np.r_[np.zeros(n_out, dtype=int), np.ones(n_in, dtype=int)]
    idx = rng.permutation(n_samples)
    return X[idx], y[idx]

def make_circles(n_samples=200, factor=0.5, noise=0.05, random_state=None):
    """A large circle containing a smaller circle."""
    rng = np.random.RandomState(random_state)
    n_out = n_samples // 2
    n_in = n_samples - n_out
    t_out = rng.uniform(0, 2 * np.pi, n_out)
    t_in = rng.uniform(0, 2 * np.pi, n_in)
    X = np.vstack([np.c_[np.cos(t_out), np.sin(t_out)],
                   factor * np.c_[np.cos(t_in), np.sin(t_in)]])
    X += rng.normal(0, noise, size=X.shape)
    y = np.r_[np.zeros(n_out, dtype=int), np.ones(n_in, dtype=int)]
    idx = rng.permutation(n_samples)
    return X[idx], y[idx]

def train_test_split(X, y, test_size=0.25, random_state=None):
    """Shuffle and split arrays into train and test subsets."""
    rng = np.random.RandomState(random_state)
    idx = rng.permutation(len(X))
    n_test = int(round(len(X) * test_size))
    test, train = idx[:n_test], idx[n_test:]
    return X[train], X[test], y[train], y[test]

def accuracy_score(y_true, y_pred):
    return np.mean(np.asarray(y_true) == np.asarray(y_pred))

def plot_decision_boundary(model, X, y, title="", ax=None, resolution=150):
    """Colour the plane by the model's prediction and overlay the data."""
    ax = ax or plt.gca()
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution),
                         np.linspace(y_min, y_max, resolution))
    Z = np.asarray(model.predict(np.c_[xx.ravel(), yy.ravel()]), dtype=float).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=25)
    ax.set_title(title)

## Part 1 — Linear SVM: sub-gradient descent on the hinge loss

In [ ]:
class LinearSVM:
    """Minimises  lam/2 ||w||^2 + mean(hinge loss)  with batch sub-gradient descent.
    (Equivalent to the C-form above with C = 1 / (lam * n).)"""

    def __init__(self, lam=0.01, lr=0.1, n_iters=1000):
        self.lam, self.lr, self.n_iters = lam, lr, n_iters

    def fit(self, X, y):
        y = np.where(y <= 0, -1, 1)
        n, d = X.shape
        self.w, self.b = np.zeros(d), 0.0
        self.loss_history = []
        for _ in range(self.n_iters):
            margins = y * (X @ self.w + self.b)
            self.loss_history.append(self.lam / 2 * self.w @ self.w + np.maximum(0, 1 - margins).mean())
            viol = margins < 1                       # points inside the margin or misclassified
            dw = self.lam * self.w - (y[viol, None] * X[viol]).sum(axis=0) / n
            db = -y[viol].sum() / n
            self.w -= self.lr * dw
            self.b -= self.lr * db
        return self

    def decision_function(self, X):
        return X @ self.w + self.b

    def predict(self, X):
        return (self.decision_function(X) >= 0).astype(int)

In [ ]:
X, y = make_blobs(n_samples=200, centers=[[-2, -2], [2, 2]], cluster_std=1.1, random_state=6)
lsvm = LinearSVM(lam=0.01, lr=0.1, n_iters=1000).fit(X, y)
print("w =", lsvm.w, " b =", round(lsvm.b, 3), " train acc =", accuracy_score(y, lsvm.predict(X)))

def plot_margin(model, X, y, ax, title):
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200),
                         np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 200))
    Z = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z > 0, alpha=0.2, cmap="coolwarm")
    ax.contour(xx, yy, Z, levels=[-1, 0, 1], colors="k", linestyles=["--", "-", "--"])
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=25)
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
plot_margin(lsvm, X, y, axes[0], "Linear SVM (sub-gradient): boundary and margins")
axes[1].plot(lsvm.loss_history); axes[1].set_xlabel("iteration"); axes[1].set_ylabel("objective")
axes[1].set_title("Training objective")
plt.tight_layout(); plt.show()

## Part 2 — Kernel SVM with SMO

In [ ]:
def linear_kernel(A, B, **_):
    return A @ B.T

def polynomial_kernel(A, B, degree=3, coef0=1.0, **_):
    return (A @ B.T + coef0) ** degree

def rbf_kernel(A, B, gamma=1.0, **_):
    sq = (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None, :] - 2 * A @ B.T
    return np.exp(-gamma * np.maximum(sq, 0))

KERNELS = {"linear": linear_kernel, "poly": polynomial_kernel, "rbf": rbf_kernel}


class SVM:
    """Soft-margin kernel SVM trained with simplified SMO (Platt 1998; CS229 notes)."""

    def __init__(self, C=1.0, kernel="rbf", tol=1e-3, max_passes=10, max_iter=10_000,
                 random_state=0, **kernel_params):
        self.C, self.tol, self.max_passes, self.max_iter = C, tol, max_passes, max_iter
        self.kernel_name = kernel
        self.kernel = lambda A, B: KERNELS[kernel](A, B, **kernel_params)
        self.rng = np.random.RandomState(random_state)

    def fit(self, X, y):
        y = np.where(y <= 0, -1.0, 1.0)
        n = len(y)
        K = self.kernel(X, X)
        alpha, b = np.zeros(n), 0.0
        passes = it = 0
        while passes < self.max_passes and it < self.max_iter:
            it += 1
            changed = 0
            for i in range(n):
                E_i = (alpha * y) @ K[:, i] + b - y[i]
                # pick i if it violates the KKT conditions
                if (y[i] * E_i < -self.tol and alpha[i] < self.C) or (y[i] * E_i > self.tol and alpha[i] > 0):
                    j = self.rng.randint(n - 1)
                    j += j >= i                                   # random j != i
                    E_j = (alpha * y) @ K[:, j] + b - y[j]
                    a_i_old, a_j_old = alpha[i], alpha[j]

                    if y[i] != y[j]:
                        L, H = max(0, a_j_old - a_i_old), min(self.C, self.C + a_j_old - a_i_old)
                    else:
                        L, H = max(0, a_i_old + a_j_old - self.C), min(self.C, a_i_old + a_j_old)
                    if L == H:
                        continue
                    eta = 2 * K[i, j] - K[i, i] - K[j, j]
                    if eta >= 0:
                        continue

                    alpha[j] = np.clip(a_j_old - y[j] * (E_i - E_j) / eta, L, H)
                    if abs(alpha[j] - a_j_old) < 1e-5:
                        continue
                    alpha[i] = a_i_old + y[i] * y[j] * (a_j_old - alpha[j])

                    b1 = b - E_i - y[i] * (alpha[i] - a_i_old) * K[i, i] - y[j] * (alpha[j] - a_j_old) * K[i, j]
                    b2 = b - E_j - y[i] * (alpha[i] - a_i_old) * K[i, j] - y[j] * (alpha[j] - a_j_old) * K[j, j]
                    if 0 < alpha[i] < self.C:
                        b = b1
                    elif 0 < alpha[j] < self.C:
                        b = b2
                    else:
                        b = (b1 + b2) / 2
                    changed += 1
            passes = passes + 1 if changed == 0 else 0

        sv = alpha > 1e-6
        self.support_vectors_, self.alpha_sv, self.y_sv = X[sv], alpha[sv], y[sv]
        self.b = b
        return self

    def decision_function(self, X):
        return self.kernel(X, self.support_vectors_) @ (self.alpha_sv * self.y_sv) + self.b

    def predict(self, X):
        return (self.decision_function(X) >= 0).astype(int)

In [ ]:
svm_lin = SVM(C=1.0, kernel="linear").fit(X, y)
print("Linear-kernel SMO: support vectors =", len(svm_lin.support_vectors_),
      " train acc =", accuracy_score(y, svm_lin.predict(X)))

fig, ax = plt.subplots(figsize=(7, 5))
plot_margin(svm_lin, X, y, ax, "SMO linear SVM — circled points are support vectors")
ax.scatter(*svm_lin.support_vectors_.T, s=150, facecolors="none", edgecolors="lime", linewidths=2)
plt.show()

### Effect of $C$
Small $C$ → wide margin, many support vectors, tolerant of errors. Large $C$ → narrow margin, fits training data tightly.

In [ ]:
Xo, yo = make_blobs(n_samples=200, centers=[[-1.5, -1.5], [1.5, 1.5]], cluster_std=1.4, random_state=2)
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, C_val in zip(axes, [0.01, 1, 100]):
    m = SVM(C=C_val, kernel="linear").fit(Xo, yo)
    plot_margin(m, Xo, yo, ax, f"C={C_val}: {len(m.support_vectors_)} SVs, acc={accuracy_score(yo, m.predict(Xo)):.2f}")
    ax.scatter(*m.support_vectors_.T, s=120, facecolors="none", edgecolors="lime", linewidths=1.5)
plt.tight_layout(); plt.show()

### Non-linear data: kernels

In [ ]:
Xc, yc = make_circles(n_samples=200, factor=0.4, noise=0.1, random_state=1)
Xmn, ymn = make_moons(n_samples=200, noise=0.15, random_state=1)

configs = [("linear", {}), ("poly", {"degree": 3}), ("rbf", {"gamma": 1.0})]
fig, axes = plt.subplots(2, 3, figsize=(17, 9.5))
for row, (Xd, yd, name) in enumerate([(Xc, yc, "circles"), (Xmn, ymn, "moons")]):
    Xtr, Xte, ytr, yte = train_test_split(Xd, yd, test_size=0.3, random_state=0)
    for col, (kernel, params) in enumerate(configs):
        m = SVM(C=1.0, kernel=kernel, **params).fit(Xtr, ytr)
        acc = accuracy_score(yte, m.predict(Xte))
        plot_margin(m, Xtr, ytr, axes[row, col], f"{name}: {kernel} kernel, test acc={acc:.2f}")
plt.tight_layout(); plt.show()

### RBF $\gamma$: the kernel width
Large $\gamma$ → very local influence → wiggly, overfit boundary.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, g in zip(axes, [0.1, 1, 50]):
    m = SVM(C=1.0, kernel="rbf", gamma=g).fit(Xmn, ymn)
    plot_margin(m, Xmn, ymn, ax, f"gamma={g}: {len(m.support_vectors_)} support vectors")
plt.tight_layout(); plt.show()

## Summary
* SVMs maximise the margin; only support vectors define the boundary.
* $C$ controls regularisation; kernels (with $\gamma$, degree) control the shape of the boundary.
* SVMs are sensitive to feature scaling — standardise inputs in practice.
* Multiclass SVMs are built from binary ones (one-vs-rest or one-vs-one).